# XBRL US API - Merged fact and report details by report.entry-url 

### This notebook returns report and fact details for specified `report.entry-url` values.

**Authenticate for access token** - [get a free XBRL US account and request provisioning for the XBRL API](https://xbrl.us/access-token), then click the run button below to execute the cell's code. Enter your XBRL US Web account email, account password, Client ID, and secret as noted, pressing the Enter key on the keyboard after each entry. 

This script loops to collect all data from the Public Filings Database. XBRL US limits records returned improve efficiency and **non-Members might not be able to return all data for a query**. Join XBRL US for comprehensive access - https://xbrl.us/join.

In [1]:
# @title
import os, re, sys, json
import requests
import pandas as pd
from IPython.display import display, HTML
import numpy as np
import getpass
from datetime import datetime
import urllib
from urllib.parse import urlencode


class tokenInfoClass:
    access_token = None
    refresh_token = None
    email = None
    username = None
    client_id = None
    client_secret = None
    url = 'https://api.xbrl.us/oauth2/token'
    headers = {"Content-Type": "application/x-www-form-urlencoded"}

def refresh(info):
    refresh_auth = {
                'client_id': info.client_id,
                'client_secret' : info.client_secret,
                'grant_type' : 'refresh_token',
                'platform' : 'ipynb',
                'refresh_token' : info.refresh_token
                }
    refreshres = requests.post(info.url, data=refresh_auth, headers=info.headers)
    refresh_json = refreshres.json()
    info.access_token = refresh_json.get('access_token')
    info.refresh_token = refresh_json.get('refresh_token')
    print('Your access token(%s) is refreshed for 60 minutes. If it expires again, run this cell to generate a new token and continue to use the query cells below.' % (info.access_token))
    return info

tokenInfo = tokenInfoClass()

# Helper to prompt only if value is missing
def prompt_if_missing(value, prompt_text, secret=False):
    if value:
        return value
    if secret:
        return getpass.getpass(prompt=prompt_text)
    return input(prompt_text)

# Load credentials (if .json exists)
creds = {}
if os.path.exists('creds.json'):
    try:
        with open('creds.json', 'r') as f:
            creds = json.load(f)
        print("Loaded .json")
    except Exception as e:
        print("Warning: failed to read from .json:", e)

if creds:
    # Try nested prod object first
    selected = None
    if isinstance(creds.get('prod'), dict):
        selected = creds['prod']
    
    # Next, try prod-prefixed keys
    if not selected:
        selected = {}
        keys = ['email', 'password', 'client_id', 'client_secret']
        for k in keys:
            prefixed_key = 'prod' + k
            if creds.get(prefixed_key):
                selected[k] = creds.get(prefixed_key)
            # fall back to top-level key if prod variant not found
            elif creds.get(k):
                selected[k] = creds.get(k)
    
    # Verify we have all required keys
    if not all(selected.get(k) for k in ('email', 'password', 'client_id', 'client_secret')):
        # Fill in missing values from prompts
        selected = {
            'email': selected.get('email'),
            'password': selected.get('password'),
            'client_id': selected.get('client_id'),
            'client_secret': selected.get('client_secret')
        }
    
    # Assign values, prompting for any missing ones
    tokenInfo.email = prompt_if_missing(selected.get('email'), 'Enter your XBRL US Web account email: ')
    tokenInfo.password = prompt_if_missing(selected.get('password'), 'Password: ', secret=True)
    tokenInfo.client_id = prompt_if_missing(selected.get('client_id'), 'Client ID: ', secret=True)
    tokenInfo.client_secret = prompt_if_missing(selected.get('client_secret'), 'Secret: ', secret=True)

    print('Using credentials from .json as available.')
else:
    # No creds.json — prompt the user
    tokenInfo.email = input('Enter your XBRL US Web account email: ')
    tokenInfo.password = getpass.getpass(prompt='Password: ')
    tokenInfo.client_id = getpass.getpass(prompt='Client ID: ')
    tokenInfo.client_secret = getpass.getpass(prompt='Secret: ')

body_auth = {'username' : tokenInfo.email,
            'client_id': tokenInfo.client_id,
            'client_secret' : tokenInfo.client_secret,
            'password' : tokenInfo.password,
            'grant_type' : 'password',
            'platform' : 'ipynb' }

#print(body_auth)

payload = urlencode(body_auth)
res = requests.request("POST", tokenInfo.url, data=payload, headers=tokenInfo.headers)
auth_json = res.json()

if 'error' in auth_json:
    print("\n\nThere was a problem generating the access token: %s  Run the first cell again and enter the credentials." % (auth_json.get('error_description', auth_json)))
else:
    tokenInfo.access_token = auth_json.get('access_token')
    tokenInfo.refresh_token = auth_json.get('refresh_token')
    if tokenInfo.access_token and tokenInfo.refresh_token:
        print ("\n\nYour access token expires in 60 minutes. After it expires, it should be regenerated automatically.  If not, run the cell rerun the first query cell. \n\nFor now, skip ahead to the section 'Make a Query'.")
    else:
        print("\n\nAuthentication completed but tokens were not returned. Response: {}".format(auth_json))

#print(vars(tokenInfo))
if tokenInfo.access_token and tokenInfo.refresh_token:
    print('\n\naccess token: ' + tokenInfo.access_token + ' refresh token: ' + tokenInfo.refresh_token)
else:
    print('\n\nNo access token was generated. Check the messages above for errors.')



Your access token expires in 60 minutes. After it expires, it should be regenerated automatically.  If not, run the cell rerun the first query cell. 

For now, skip ahead to the section 'Make a Query'.


access token: b68b860a-6836-46b6-a350-99b8788aad4b refresh token: 138fd87f-f247-459c-ba6a-9e342b2fca21


# Define search filters and fields to return

The section below defines parameters for the endpoint to evaluate `keyword strings` for the variable defined. Click in the code cell then click the run button to save the parameters, then run the next cell to query and return attributes for each concept.

In [2]:
### Define the parameters for the filter and fields to be returned

# Define endpoint (common values: fact, entity, report, cube, label, concept, relationship - see https://xbrlus/github.io/xbrl-api for additional endpoint options)

endpoint = 'report/fact'

concept = ['EntityRegistrantName']

kwl_variable = 'report.entry-url'

Keyword_List = [
  'http://www.sec.gov/Archives/edgar/data/78239/000007823925000018/pvh-20250202.htm',
  'http://www.sec.gov/Archives/edgar/data/1166663/000119312525079101/d896344d20f.htm',
  'http://www.sec.gov/Archives/edgar/data/1059142/000095017024092308/ghi-20240630.htm',
  'http://www.sec.gov/Archives/edgar/data/1551901/000155837025002182/scm-20241231x10k.htm',
  'http://www.sec.gov/Archives/edgar/data/1870267/000155837024011897/tmb-20240630x10q.htm',
  'http://www.sec.gov/Archives/edgar/data/1551901/000155837025002182/scm-20241231x10k.htm',
  'http://www.sec.gov/Archives/edgar/data/1870267/000155837024015521/tmb-20240930x10q.htm',
  'http://www.sec.gov/Archives/edgar/data/1040971/000104097124000031/slg-20240930.htm',
  'http://www.sec.gov/Archives/edgar/data/1018979/000095017025030059/amsf-20241231.htm',
  'http://www.sec.gov/Archives/edgar/data/1023024/000102302424000085/anip-20240630.htm',
  'http://www.sec.gov/Archives/edgar/data/1720446/000141057825000903/zepp-20241231x20f.htm',
  'http://www.sec.gov/Archives/edgar/data/1010134/000117891325001749/zk2533076.htm',
  'http://www.sec.gov/Archives/edgar/data/2000640/000121390025024746/ea0234430-s1_damon.htm',
  'http://www.sec.gov/Archives/edgar/data/1589149/000164117225004803/form10-k.htm',
  'http://www.sec.gov/Archives/edgar/data/1176948/000162828025008665/ares-20241231.htm',
  'http://www.sec.gov/Archives/edgar/data/1758730/000175873025000038/tw-20241231.htm',
  'http://www.sec.gov/Archives/edgar/data/745732/000074573225000010/rost-20250201.htm',
  'http://www.sec.gov/Archives/edgar/data/1075736/000165495424010039/cxdo_10q.htm',
  'http://www.sec.gov/Archives/edgar/data/1360565/000149315225007682/form10-k.htm',
  'http://www.sec.gov/Archives/edgar/data/1043337/000104333725000012/sri-20241231.htm',
  'http://www.sec.gov/Archives/edgar/data/1479290/000147929024000111/rvnc-20240630.htm',
  'http://www.sec.gov/Archives/edgar/data/1309108/000130910825000017/wex-20241231.htm',
  'http://www.sec.gov/Archives/edgar/data/1386716/000095015725000241/sblk-20241231.htm',
  'http://www.sec.gov/Archives/edgar/data/1745916/000155837025001148/pfsi-20241231x10k.htm',
  'http://www.sec.gov/Archives/edgar/data/1857086/000149315224032270/form10-q.htm',
  'http://www.sec.gov/Archives/edgar/data/1143513/000114351324000011/glad-20240930.htm',
  'http://www.sec.gov/Archives/edgar/data/1512931/000151293125000013/mrcc-20241231.htm',
  'http://www.sec.gov/Archives/edgar/data/1534254/000153425425000004/cion-20241231.htm',
  'http://www.sec.gov/Archives/edgar/data/1383414/000095017024130670/pnnt-20240930.htm',
  'http://www.sec.gov/Archives/edgar/data/1496099/000149609925000010/nmfc-20241231.htm',
  'http://www.sec.gov/Archives/edgar/data/1002910/000100291024000111/aee-20240930.htm',
  'http://www.sec.gov/Archives/edgar/data/1280784/000128078424000020/htgc-20240630.htm',
  'http://www.sec.gov/Archives/edgar/data/1415332/000164033425000577/ltum_10k.htm',
  'http://www.sec.gov/Archives/edgar/data/1002910/000100291024000111/aee-20240930.htm',
  'http://www.sec.gov/Archives/edgar/data/106535/000095017025021254/wy-20241231.htm',
  'http://www.sec.gov/Archives/edgar/data/1280784/000128078424000020/htgc-20240630.htm',
  'http://www.sec.gov/Archives/edgar/data/1415332/000164033425000577/ltum_10k.htm',
  'http://www.sec.gov/Archives/edgar/data/1813744/000159991624000292/worldscan_q224f.htm',
  'http://www.sec.gov/Archives/edgar/data/1001316/000143774925005908/tgtx20241231_10k.htm',
  'http://www.sec.gov/Archives/edgar/data/1653558/000165355824000144/prth-20240930.htm',
  'http://www.sec.gov/Archives/edgar/data/1015155/000114036125012058/ef20029694_10k.htm',
  'http://www.sec.gov/Archives/edgar/data/1740742/000182912625002700/unicoininc_10k.htm',
  'http://www.sec.gov/Archives/edgar/data/1034760/000165495425004306/wyy_10k.htm',
  'http://www.sec.gov/Archives/edgar/data/1287750/000128775025000020/arcc-20250331.htm',
  'http://www.sec.gov/Archives/edgar/data/1287750/000128775025000020/arcc-20250331.htm',
  'http://www.sec.gov/Archives/edgar/data/1005286/000100528625000055/lfcr-20250223.htm',
  'http://www.sec.gov/Archives/edgar/data/1018164/000101816425000027/wlfc-20241231.htm',
  'http://www.sec.gov/Archives/edgar/data/1015922/000117891325000867/zk2532836.htm',
  'http://www.sec.gov/Archives/edgar/data/1018840/000101884025000013/anf-20250201.htm',
  'http://www.sec.gov/Archives/edgar/data/1124804/000095017025041181/mdrx-20221231.htm',
  'http://www.sec.gov/Archives/edgar/data/1094366/000117891325001101/zk2532903.htm',
  'http://www.sec.gov/Archives/edgar/data/1127475/000118518525000304/dbmm10q022825.htm',
  'http://www.sec.gov/Archives/edgar/data/1127475/000118518525000304/dbmm10q022825.htm',
  'http://www.sec.gov/Archives/edgar/data/1127475/000118518525000304/dbmm10q022825.htm',
  'http://www.sec.gov/Archives/edgar/data/1692376/000095017025037433/vel-20241231.htm',
  'http://www.sec.gov/Archives/edgar/data/101538/000165495425003048/uamy_10k.htm',
  'http://www.sec.gov/Archives/edgar/data/1000209/000095017025038693/mfin-20241231.htm',
  'http://www.sec.gov/Archives/edgar/data/1009829/000118518525000170/jakk10k123124.htm',
  'http://www.sec.gov/Archives/edgar/data/1000209/000095017025038693/mfin-20241231.htm',
  'http://www.sec.gov/Archives/edgar/data/1009829/000118518525000170/jakk10k123124.htm',
  'http://www.sec.gov/Archives/edgar/data/1269026/000149315225010870/form10-k.htm',
  'http://www.sec.gov/Archives/edgar/data/1004724/000095017025047742/rhe-20241231.htm',
  'http://www.sec.gov/Archives/edgar/data/1010134/000117891324004016/zk2432438.htm',
  'http://www.sec.gov/Archives/edgar/data/1512228/000153949725000278/n2574_x244-def14a.htm',
  'http://www.sec.gov/Archives/edgar/data/1010134/000117891324004016/zk2432438.htm',
  'http://www.sec.gov/Archives/edgar/data/1004724/000095017024096210/rhe-20240630.htm',
  'http://www.sec.gov/Archives/edgar/data/1699039/000169903924000095/rngr-20240630.htm',
  'http://www.sec.gov/Archives/edgar/data/1001385/000143774925005465/nwpx20241231_10k.htm',
  'http://www.sec.gov/Archives/edgar/data/100885/000143774924023463/unp20240630c_10q.htm',
  'http://www.sec.gov/Archives/edgar/data/1006837/000100683724000122/vate-20240630.htm',
  'http://www.sec.gov/Archives/edgar/data/1104485/000110448524000147/nog-20240630.htm',
  'http://www.sec.gov/Archives/edgar/data/1000694/000100069424000035/nvax-20240630.htm',
  'http://www.sec.gov/Archives/edgar/data/1004980/000100498024000107/pcg-20240630.htm',
  'http://www.sec.gov/Archives/edgar/data/1015922/000117891325000867/zk2532836.htm',
  'http://www.sec.gov/Archives/edgar/data/1030192/000165495424013653/njmc_10q.htm',
  'http://www.sec.gov/Archives/edgar/data/101382/000095017024088982/umbf-20240630.htm',
  'http://www.sec.gov/Archives/edgar/data/1000045/000095017024100547/nick-20240630.htm',
  'http://www.sec.gov/Archives/edgar/data/1005284/000095017024089283/oled-20240630.htm',
  'http://www.sec.gov/Archives/edgar/data/1370755/000095017024092283/tcpc-20240630.htm',
  'http://www.sec.gov/Archives/edgar/data/1037540/000165642325000009/bxp-20241231.htm',
  'http://www.sec.gov/Archives/edgar/data/1004980/000100498024000107/pcg-20240630.htm',
  'http://www.sec.gov/Archives/edgar/data/106640/000010664024000125/whr-20240630.htm',
  'http://www.sec.gov/Archives/edgar/data/1046025/000104602524000164/hfwa-20240630.htm',
  'http://www.sec.gov/Archives/edgar/data/1018164/000101816424000066/wlfc-20240630.htm',
  'http://www.sec.gov/Archives/edgar/data/1430725/000114036124035804/gsl-20240630.htm',
  'http://www.sec.gov/Archives/edgar/data/1046025/000104602524000164/hfwa-20240630.htm',
  'http://www.sec.gov/Archives/edgar/data/1499961/000182912624007343/mullenautomotive_s1a.htm',
  'http://www.sec.gov/Archives/edgar/data/1499961/000182912624007343/mullenautomotive_s1a.htm',
  'http://www.sec.gov/Archives/edgar/data/1006837/000100683724000122/vate-20240630.htm',
  'http://www.sec.gov/Archives/edgar/data/1012100/000162828024036175/see-20240630.htm',
  'http://www.sec.gov/Archives/edgar/data/1047340/000104734024000202/fdp-20240628.htm',
  'http://www.sec.gov/Archives/edgar/data/1005286/000100528624000110/lfcr-20230827.htm',
  'http://www.sec.gov/Archives/edgar/data/100790/000002991524000009/ucc-20240630.htm',
  'http://www.sec.gov/Archives/edgar/data/1001907/000143774924032014/astc20241017_def14a.htm',
  'http://www.sec.gov/Archives/edgar/data/1004702/000100470224000098/ocfc-20240630.htm',
  'http://www.sec.gov/Archives/edgar/data/1133421/000113342124000052/noc-20240630.htm',
  'http://www.sec.gov/Archives/edgar/data/1000230/000143774924038248/ex_755614.htm',
  'http://www.sec.gov/Archives/edgar/data/1378789/000137878924000030/aer-20231231.htm',
  'http://www.sec.gov/Archives/edgar/data/1026214/000102621425000040/fmcc-20241231.htm',
  'http://www.sec.gov/Archives/edgar/data/101538/000165495425003048/uamy_10k.htm',
  'http://www.sec.gov/Archives/edgar/data/1034760/000165495425004306/wyy_10k.htm',
  'http://www.sec.gov/Archives/edgar/data/1002910/000100291024000111/aee-20240930.htm',
  'http://www.sec.gov/Archives/edgar/data/1002910/000100291024000111/aee-20240930.htm',
  'http://www.sec.gov/Archives/edgar/data/1002910/000100291024000111/aee-20240930.htm',
  'http://www.sec.gov/Archives/edgar/data/1009922/000165495425002241/nxt_20f.htm',
  'http://www.sec.gov/Archives/edgar/data/1061164/000147793224004368/pgid_10q.htm',
  'http://www.sec.gov/Archives/edgar/data/1941506/000173112225000613/e6523_20f.htm',
  'http://www.sec.gov/Archives/edgar/data/100493/000010049324000103/tsn-20240629.htm',
  'http://www.sec.gov/Archives/edgar/data/1133421/000113342125000023/noc-20250331.htm',
  'http://www.sec.gov/Archives/edgar/data/1050140/000092963825001603/a40f.htm',
  'http://www.sec.gov/Archives/edgar/data/1037540/000165642324000046/bxp-20240630.htm',
  'http://www.sec.gov/Archives/edgar/data/1003815/000141057824001261/bctciv-20240630x10q.htm',
  'http://www.sec.gov/Archives/edgar/data/101538/000165495424014176/uamy_10q.htm',
  'http://www.sec.gov/Archives/edgar/data/101538/000165495424010214/uamy_10q.htm',
  'http://www.sec.gov/Archives/edgar/data/1024672/000117891325001230/zk2532968.htm',
  'http://www.sec.gov/Archives/edgar/data/1633978/000162828024038024/lite-20240629.htm',
  'http://www.sec.gov/Archives/edgar/data/1853044/000182912624006527/aeriestech_10k.htm',
  'http://www.sec.gov/Archives/edgar/data/1056358/000105635825000018/mtex-20241231.htm',
  'http://www.sec.gov/Archives/edgar/data/103730/000010373024000122/vishayintertech_10q.htm',
  'http://www.sec.gov/Archives/edgar/data/1015922/000117891325000867/zk2532836.htm',
  'http://www.sec.gov/Archives/edgar/data/100493/000010049324000103/tsn-20240629.htm',
  'http://www.sec.gov/Archives/edgar/data/100493/000010049324000103/tsn-20240629.htm',
  'http://www.sec.gov/Archives/edgar/data/1061069/000143774924015406/awx20240331_10q.htm',
  'http://www.sec.gov/Archives/edgar/data/1004989/000143774924035313/sgrp20240930_10q.htm',
  'http://www.sec.gov/Archives/edgar/data/1495648/000147793225002903/bdcc_10q.htm',
  'http://www.sec.gov/Archives/edgar/data/1037540/000165642324000046/bxp-20240630.htm',
  'http://www.sec.gov/Archives/edgar/data/1850391/000121465925004851/wtb3242510k.htm',
  'http://www.sec.gov/Archives/edgar/data/1084133/000126493124000027/real10q124.htm',
  'http://www.sec.gov/Archives/edgar/data/1030192/000165495424009573/njmc_10q.htm',
  'http://www.sec.gov/Archives/edgar/data/1001115/000143774924025790/geos20240630_10q.htm',
  'http://www.sec.gov/Archives/edgar/data/1026214/000102621424000068/fmcc-20240630.htm',
  'http://www.sec.gov/Archives/edgar/data/1005284/000095017024118989/oled-20240930.htm',
  'http://www.sec.gov/Archives/edgar/data/1098146/000162828024048517/pnbk-20240930.htm',
  'http://www.sec.gov/Archives/edgar/data/1787414/000143774925009807/bsbk20241231_10k.htm',
  'http://www.sec.gov/Archives/edgar/data/1098146/000162828024048517/pnbk-20240930.htm',
  'http://www.sec.gov/Archives/edgar/data/1163389/000190359624000204/npbs_10k.htm',
  'http://www.sec.gov/Archives/edgar/data/1002590/000095017025014482/sgu-20241231.htm',
  'http://www.sec.gov/Archives/edgar/data/1015922/000117891325000867/zk2532836.htm',
  'http://www.sec.gov/Archives/edgar/data/1015922/000117891325000867/zk2532836.htm',
  'http://www.sec.gov/Archives/edgar/data/1411342/000141134225000015/efc-20241231.htm',
  'http://www.sec.gov/Archives/edgar/data/1499961/000182912624007343/mullenautomotive_s1a.htm',
  'http://www.sec.gov/Archives/edgar/data/1339688/000106299324015240/form10q.htm',
  'http://www.sec.gov/Archives/edgar/data/1037540/000165642324000046/bxp-20240630.htm',
  'http://www.sec.gov/Archives/edgar/data/1331612/000147793224006452/imth_10k.htm',
  'http://www.sec.gov/Archives/edgar/data/1035201/000103520125000003/cwt-20241231.htm',
  'http://www.sec.gov/Archives/edgar/data/1018281/000143774924030004/itkg20240630_10k.htm',
  'http://www.sec.gov/Archives/edgar/data/1552275/000155227524000056/sun-20240630.htm',
  'http://www.sec.gov/Archives/edgar/data/1002590/000095017025014482/sgu-20241231.htm',
  'http://www.sec.gov/Archives/edgar/data/1015922/000117891325000867/zk2532836.htm',
  'http://www.sec.gov/Archives/edgar/data/1037540/000165642325000009/bxp-20241231.htm',
  'http://www.sec.gov/Archives/edgar/data/1109138/000117891325000945/zk2532834.htm',
  'http://www.sec.gov/Archives/edgar/data/1056358/000105635825000018/mtex-20241231.htm',
  'http://www.sec.gov/Archives/edgar/data/1170010/000117001025000024/kmx-20250228.htm',
  'http://www.sec.gov/Archives/edgar/data/1009922/000165495425002241/nxt_20f.htm',
  'http://www.sec.gov/Archives/edgar/data/1037646/000103764624000012/mtd-20240331.htm',
  'http://www.sec.gov/Archives/edgar/data/1009922/000165495425002241/nxt_20f.htm',
  'http://www.sec.gov/Archives/edgar/data/1291855/000117891325001468/zk2533057.htm',
  'http://www.sec.gov/Archives/edgar/data/1051470/000105147025000089/cci-20241231.htm',
  'http://www.sec.gov/Archives/edgar/data/100517/000010051725000091/ual-20250331.htm',
  'http://www.sec.gov/Archives/edgar/data/1022652/000168316824008826/inseego_s1.htm',
  'http://www.sec.gov/Archives/edgar/data/1474167/000147793224004545/cosm_10k.htm',
  'http://www.sec.gov/Archives/edgar/data/1661039/000165495424008305/tptw_s1.htm',
  'http://www.sec.gov/Archives/edgar/data/1309082/000147793224005266/cei_10ka.htm',
  'http://www.sec.gov/Archives/edgar/data/1040470/000165495425004120/aehr_10q.htm',
  'http://www.sec.gov/Archives/edgar/data/1145604/000182646624000120/form-10q.htm',
  'http://www.sec.gov/Archives/edgar/data/1464865/000147793224004240/accredited_10k.htm',
  'http://www.sec.gov/Archives/edgar/data/1040470/000165495424009642/aehr_10k.htm',
  'http://www.sec.gov/Archives/edgar/data/1467761/000182912624003547/miniminc_10q.htm',
  'http://www.sec.gov/Archives/edgar/data/1037540/000165642325000009/bxp-20241231.htm',
  'http://www.sec.gov/Archives/edgar/data/1420368/000147793224004253/dlti_10q.htm',
  'http://www.sec.gov/Archives/edgar/data/1040470/000165495424009642/aehr_10k.htm',
  'http://www.sec.gov/Archives/edgar/data/1099369/000106299324019439/form10k.htm',
  'http://www.sec.gov/Archives/edgar/data/1075736/000165495424010039/cxdo_10q.htm',
  'http://www.sec.gov/Archives/edgar/data/101538/000165495424014176/uamy_10q.htm',
  'http://www.sec.gov/Archives/edgar/data/1026214/000102621424000068/fmcc-20240630.htm',
  'http://www.sec.gov/Archives/edgar/data/100493/000010049324000103/tsn-20240629.htm',
  'http://www.sec.gov/Archives/edgar/data/1394056/000095017024121839/oss-20240930.htm',
  'http://www.sec.gov/Archives/edgar/data/1684693/000117891325001497/zk2533067.htm',
  'http://www.sec.gov/Archives/edgar/data/1266806/000175392624000959/g084215_10q.htm',
  'http://www.sec.gov/Archives/edgar/data/1004724/000095017024096210/rhe-20240630.htm',
  'http://www.sec.gov/Archives/edgar/data/1081938/000165495425004122/canna_10k.htm',
  'http://www.sec.gov/Archives/edgar/data/1037646/000103764624000034/mtd-20240930.htm',
  'http://www.sec.gov/Archives/edgar/data/7789/000000778925000013/asb-20241231.htm',
  'http://www.sec.gov/Archives/edgar/data/1362988/000162828024018039/ayr-20240229.htm',
  'http://www.sec.gov/Archives/edgar/data/1418819/000162828025005302/irdm-20241231.htm',
  'http://www.sec.gov/Archives/edgar/data/1051470/000105147025000089/cci-20241231.htm',
  'http://www.sec.gov/Archives/edgar/data/1726711/000121390024045191/ea0206431-10q_aditxtinc.htm',
  'http://www.sec.gov/Archives/edgar/data/1015922/000117891325000867/zk2532836.htm',
  'http://www.sec.gov/Archives/edgar/data/1015922/000117891325000867/zk2532836.htm',
  'http://www.sec.gov/Archives/edgar/data/1015922/000117891325000867/zk2532836.htm',
  'http://www.sec.gov/Archives/edgar/data/724910/000137647424000323/nvec-20240630.htm',
  'http://www.sec.gov/Archives/edgar/data/1411342/000141134225000015/efc-20241231.htm',
  'http://www.sec.gov/Archives/edgar/data/1072725/000107272524000017/gdrzfform6kex991052924.htm',
  'http://www.sec.gov/Archives/edgar/data/1813744/000159991624000287/worldscan_q124f.htm',
  'http://www.sec.gov/Archives/edgar/data/1017655/000143774924027219/payd20240630_10q.htm',
  'http://www.sec.gov/Archives/edgar/data/1276187/000127618724000040/et-20240331.htm',
  'http://www.sec.gov/Archives/edgar/data/1009922/000165495425002241/nxt_20f.htm',
  'http://www.sec.gov/Archives/edgar/data/101538/000165495424010214/uamy_10q.htm',
  'http://www.sec.gov/Archives/edgar/data/1056358/000105635825000018/mtex-20241231.htm',
  'http://www.sec.gov/Archives/edgar/data/1037540/000165642324000046/bxp-20240630.htm',
  'http://www.sec.gov/Archives/edgar/data/1013857/000101385724000155/pega-20240630.htm',
  'http://www.sec.gov/Archives/edgar/data/1668340/000155278124000598/e24436_bctf-10q.htm',
  'http://www.sec.gov/Archives/edgar/data/1618921/000161892124000084/wba-20240831.htm',
  'http://www.sec.gov/Archives/edgar/data/1040470/000165495424009642/aehr_10k.htm',
  'http://www.sec.gov/Archives/edgar/data/1037540/000165642324000046/bxp-20240630.htm',
  'http://www.sec.gov/Archives/edgar/data/1000045/000095017024130247/omcc-20240930.htm',
  'http://www.sec.gov/Archives/edgar/data/1760903/000149315224019873/form10-q.htm',
  'http://www.sec.gov/Archives/edgar/data/1389545/000143774925010730/nby20241231_10k.htm',
  'http://www.sec.gov/Archives/edgar/data/915358/000091535824000010/sgma-20240430x10k.htm',
  'http://www.sec.gov/Archives/edgar/data/1514056/000164117225001897/form10-k.htm',
  'http://www.sec.gov/Archives/edgar/data/1106213/000119983524000190/sfrx-10q.htm',
  'http://www.sec.gov/Archives/edgar/data/1637880/000163788024000044/tris-20240630.htm',
  'http://www.sec.gov/Archives/edgar/data/1364885/000162828025009088/spr-20241231.htm',
  'http://www.sec.gov/Archives/edgar/data/101829/000010182924000027/rtx-20240630.htm',
  'http://www.sec.gov/Archives/edgar/data/1364885/000162828025009088/spr-20241231.htm',
  'http://www.sec.gov/Archives/edgar/data/1364885/000162828025009088/spr-20241231.htm',
  'http://www.sec.gov/Archives/edgar/data/1309082/000147793224005253/cei_10q.htm',
  'http://www.sec.gov/Archives/edgar/data/1367408/000159991624000317/sino_q323o.htm',
  'http://www.sec.gov/Archives/edgar/data/1431074/000139390524000172/brgo-20240331.htm',
  'http://www.sec.gov/Archives/edgar/data/1309082/000147793224005253/cei_10q.htm',
  'http://www.sec.gov/Archives/edgar/data/1276531/000137647424000469/scgy-20240630.htm',
  'http://www.sec.gov/Archives/edgar/data/1085277/000166357725000110/skvi10k123124.htm',
  'http://www.sec.gov/Archives/edgar/data/1025771/000147793224004433/cpka_10q.htm',
  'http://www.sec.gov/Archives/edgar/data/1111711/000111171124000032/nix-20240630.htm',
  'http://www.sec.gov/Archives/edgar/data/1093672/000165495424013755/pebk_10q.htm',
  'http://www.sec.gov/Archives/edgar/data/1050140/000092963824003162/form6k.htm',
  'http://www.sec.gov/Archives/edgar/data/1784700/000095017025043295/ck0001784700-20241231.htm',
  'http://www.sec.gov/Archives/edgar/data/1009922/000165495425002241/nxt_20f.htm',
  'http://www.sec.gov/Archives/edgar/data/1693317/000169331724000009/cch-20240630.htm',
  'http://www.sec.gov/Archives/edgar/data/1049011/000149315224046747/form10-q.htm',
  'http://www.sec.gov/Archives/edgar/data/55067/000162828024020015/k-20240330.htm',
  'http://www.sec.gov/Archives/edgar/data/874766/000087476624000108/hig-20240630.htm',
  'http://www.sec.gov/Archives/edgar/data/1605331/000166357724000175/abqq_s1a.htm',
  'http://www.sec.gov/Archives/edgar/data/874766/000087476624000108/hig-20240630.htm',
  'http://www.sec.gov/Archives/edgar/data/763901/000119312525043848/d924089d10k.htm',
  'http://www.sec.gov/Archives/edgar/data/874766/000087476624000108/hig-20240630.htm',
  'http://www.sec.gov/Archives/edgar/data/1023313/000095017025035158/forr-20241231.htm',
  'http://www.sec.gov/Archives/edgar/data/1886894/000164117225000784/form10-k.htm',
  'http://www.sec.gov/Archives/edgar/data/1345126/000134512624000039/codi-20240630.htm',
  'http://www.sec.gov/Archives/edgar/data/1345126/000134512625000015/codi-20241231.htm',
  'http://www.sec.gov/Archives/edgar/data/1109357/000110935725000043/exc-20241231.htm',
  'http://www.sec.gov/Archives/edgar/data/1077688/000118518525000321/hoft10k020225.htm',
  'http://www.sec.gov/Archives/edgar/data/1109357/000110935725000043/exc-20241231.htm',
  'http://www.sec.gov/Archives/edgar/data/72333/000007233325000039/jwn-20250201.htm',
  'http://www.sec.gov/Archives/edgar/data/716634/000071663425000006/rdi-20240930x10qa.htm',
  'http://www.sec.gov/Archives/edgar/data/1393311/000139331124000167/psa-20240630.htm',
  'http://www.sec.gov/Archives/edgar/data/1030192/000165495424013653/njmc_10q.htm',
  'http://www.sec.gov/Archives/edgar/data/1502557/000149315224019649/form10-q.htm',
  'http://www.sec.gov/Archives/edgar/data/1030192/000165495424013653/njmc_10q.htm',
  'http://www.sec.gov/Archives/edgar/data/1502557/000149315224019649/form10-q.htm',
  'http://www.sec.gov/Archives/edgar/data/1419051/000149315224021937/form10-q.htm',
  'http://www.sec.gov/Archives/edgar/data/1817004/000149315224022374/forms-1a.htm',
  'http://www.sec.gov/Archives/edgar/data/1086303/000121390024068246/ea0210084-10q_hongchang.htm',
  'http://www.sec.gov/Archives/edgar/data/1326160/000132616025000072/duk-20241231.htm',
  'http://www.sec.gov/Archives/edgar/data/1416090/000109690624002155/imii_10q.htm',
  'http://www.sec.gov/Archives/edgar/data/830656/000149315224035352/form10-q.htm',
  'http://www.sec.gov/Archives/edgar/data/830656/000149315224035352/form10-q.htm',
  'http://www.sec.gov/Archives/edgar/data/1825384/000141057825000437/none-20241231x10k.htm',
  'http://www.sec.gov/Archives/edgar/data/1825384/000141057825000437/none-20241231x10k.htm',
  'http://www.sec.gov/Archives/edgar/data/1037540/000165642325000009/bxp-20241231.htm',
  'http://www.sec.gov/Archives/edgar/data/1026214/000102621424000097/fmcc-20240930.htm',
  'http://www.sec.gov/Archives/edgar/data/1037540/000165642325000009/bxp-20241231.htm',
  'http://www.sec.gov/Archives/edgar/data/1022321/000102232125000023/gel-20241231.htm',
  'http://www.sec.gov/Archives/edgar/data/1831868/000095017025057332/icu-20241231.htm',
  'http://www.sec.gov/Archives/edgar/data/1911545/000121390024100296/ea0221576-10q_hanryu.htm',
  'http://www.sec.gov/Archives/edgar/data/1893448/000149315224032284/form10-q.htm',
  'http://www.sec.gov/Archives/edgar/data/1661039/000165495425001239/tptw_10ka.htm',
  'http://www.sec.gov/Archives/edgar/data/1661039/000165495425001238/tptw_10q.htm',
  'http://www.sec.gov/Archives/edgar/data/1093672/000165495424010004/pebk_10q.htm',
  'http://www.sec.gov/Archives/edgar/data/1012019/000143774924033921/rusha20240930_10q.htm',
  'http://www.sec.gov/Archives/edgar/data/1391933/000092708924000086/qnto20240331_10q.htm',
  'http://www.sec.gov/Archives/edgar/data/1345126/000134512625000015/codi-20241231.htm',
  'http://www.sec.gov/Archives/edgar/data/110471/000011047124000176/www-20240928.htm',
  'http://www.sec.gov/Archives/edgar/data/1329606/000149315224020742/form10-q.htm',
  'http://www.sec.gov/Archives/edgar/data/1346022/000106299325000717/form10q.htm',
  'http://www.sec.gov/Archives/edgar/data/19745/000162828024046369/cpk-20240930.htm',
  'http://www.sec.gov/Archives/edgar/data/1037540/000165642324000046/bxp-20240630.htm',
  'http://www.sec.gov/Archives/edgar/data/1037540/000165642324000046/bxp-20240630.htm',
  'http://www.sec.gov/Archives/edgar/data/1111711/000111171124000032/nix-20240630.htm',
  'http://www.sec.gov/Archives/edgar/data/1051470/000105147025000089/cci-20241231.htm',
  'http://www.sec.gov/Archives/edgar/data/1037646/000103764624000012/mtd-20240331.htm',
  'http://www.sec.gov/Archives/edgar/data/1653653/000165365324000013/rrr-20240630.htm',
  'http://www.sec.gov/Archives/edgar/data/4904/000000490424000100/aep-20240930.htm',
  'http://www.sec.gov/Archives/edgar/data/1056358/000105635825000018/mtex-20241231.htm',
  'http://www.sec.gov/Archives/edgar/data/1668340/000155278125000058/e25055_bctf-10k.htm',
  'http://www.sec.gov/Archives/edgar/data/2008748/000093041325000900/c111957_10k-ixbrl.htm',
  'http://www.sec.gov/Archives/edgar/data/1050915/000105091524000140/pwr-20240630.htm',
  'http://www.sec.gov/Archives/edgar/data/1037540/000165642324000046/bxp-20240630.htm',
  'http://www.sec.gov/Archives/edgar/data/787250/000078725025000012/dpl-20241231.htm',
  'http://www.sec.gov/Archives/edgar/data/1495231/000149523124000157/izea-20240630.htm',
  'http://www.sec.gov/Archives/edgar/data/1587732/000158773224000060/ogs-20240930.htm'
]

fields = [
         'report.period-end',
         'report.document-type',
         'dts.target-namespace',
         'dts-entry-point',
         'report.entry-url',
         'report.html-url',
         'report.sec-url',
         'report.base-taxonomy',
         'entity.ticker',
         'report.entity-name'
         ]

# Set unique rows as True of False (True drops any duplicate rows)
unique = True

# Limit the number of rows displayed by the notebook (does not impact the data frame)
rows_to_display = 6 # Set as '' to display all rows in the notebook

params = {
     'concept.local-name': ','.join(concept),
     kwl_variable: ','.join(Keyword_List),
     'fields': ','.join(fields)
     }

print('\n\nClick the run button below to execute this query.\n\n')



Click the run button below to execute this query.




In [3]:
# @title
### Execute the query with loop for all results
### THIS SECTION DOES NOT NEED TO BE EDITED

search_endpoint = 'https://api.xbrl.us/api/v1/' + endpoint + '/search'
if unique:
    search_endpoint += '?unique'
orig_fields = params['fields']
res_df = []
query_start = datetime.now()

import math
total_keywords = len(Keyword_List)
keyword_batch_num = 1
rounds = math.ceil(total_keywords/keyword_batch_num)
round_num = 1
total_rows = 0
your_limit = 0

for x in range(0, total_keywords, keyword_batch_num):
    offset_value = 0
    count = 0
    offset_value = 0
    printed = False
    run_query = True
    segment_query_start = datetime.now()
    params[kwl_variable] = ','.join(Keyword_List[x:x+keyword_batch_num])
    params['fields'] = orig_fields
    print('Round %d/%d keyword "%s"' % (round_num, rounds, ','.join(Keyword_List[x:x+keyword_batch_num])))
    res_df_segment = []
    #print(params)

    while True:
        if not printed:
            print('On', query_start.strftime('%c'), tokenInfo.email, '(client ID:', str(tokenInfo.client_id.split('-')[0]), '...) started the query ')
            printed = True
        retry = 0
        while retry < 3:
            res = requests.get(search_endpoint, params=params, headers={'Authorization' : 'Bearer {}'.format(tokenInfo.access_token)})
            res_json = res.json()
            if 'error' in res_json:
                if res_json['error_description'] == 'Bad or expired token':
                    tokenInfo = refresh(tokenInfo)
                else:
                    print('There was an error: {}'.format(res_json['error_description']))
                    run_query = False
                    break
            else:
                    break
            retry +=1
            if retry >= 3:
                print('Cannot refresh the access token.  Run the first query block, then rerun the query.')
                run_query = False

        if not run_query:
            break

        print('up to', str(offset_value + res_json['paging']['limit']), 'records are found so far ...')

        res_df_segment += res_json['data']
        your_limit = res_json['paging']['limit']

        if res_json['paging']['count'] < res_json['paging']['limit']:
            print(' - this set contained fewer than the', res_json['paging']['limit'], 'possible, only', str(res_json['paging']['count']), 'records.')
            break
        else:
            offset_value += res_json['paging']['limit']
            if 100 == res_json['paging']['limit']:
                    params['fields'] = orig_fields + ',' + endpoint + '.offset({})'.format(offset_value)
                    if offset_value == 10 * res_json['paging']['limit']:
                            break
            elif 500 == res_json['paging']['limit']:
                    params['fields'] = orig_fields + ',' + endpoint + '.offset({})'.format(offset_value)
                    if offset_value == 4 * res_json['paging']['limit']:
                            break
            params['fields'] = orig_fields + ',' + endpoint + '.offset({})'.format(offset_value)

    current_datetime = datetime.now().replace(microsecond=0)
    time_taken = current_datetime - segment_query_start
    #print('\nAt %s the query %s finished with  %d rows returned in %s.\n%s\n' % (current_datetime.strftime("%c"), params['label.text'], len(res_df_segment), str(time_taken), urllib.parse.unquote(res.url)))
    total_rows += len(res_df_segment)
    round_num += 1
    res_df += res_df_segment
    your_limit = res_json['paging']['limit']
    limit_message = 'If the results below match the limit noted above, you might not be seeing all rows, and should consider upgrading (https://xbrl.us/access-token).\n'

    if your_limit == 100:
        print('\nThis non-Member account has a limit of ' , 10 * your_limit, ' rows per query from our Public Filings Database. ' + limit_message)
    elif your_limit == 500:
        print('\nThis Basic Individual Member account has a limit of ', 4 * your_limit, ' rows per query from our Public Filings Database. ' + limit_message)

if not 'error' in res_json:
    current_datetime = datetime.now().replace(microsecond=0)
    all_time_taken = current_datetime - query_start
    index = pd.DataFrame(res_df).index
    all_rows = len(index)
    
    df = pd.DataFrame(res_df).sort_values(by=['report.entry-url'])

df.head(10)

Round 1/291 keyword "http://www.sec.gov/Archives/edgar/data/78239/000007823925000018/pvh-20250202.htm"
On Tue Apr 21 12:30:35 2026 test.tauriello@xbrl.us (client ID: 69e1257c ...) started the query 
up to 5000 records are found so far ...
 - this set contained fewer than the 5000 possible, only 1 records.
Round 2/291 keyword "http://www.sec.gov/Archives/edgar/data/1166663/000119312525079101/d896344d20f.htm"
On Tue Apr 21 12:30:35 2026 test.tauriello@xbrl.us (client ID: 69e1257c ...) started the query 
up to 5000 records are found so far ...
 - this set contained fewer than the 5000 possible, only 1 records.
Round 3/291 keyword "http://www.sec.gov/Archives/edgar/data/1059142/000095017024092308/ghi-20240630.htm"
On Tue Apr 21 12:30:35 2026 test.tauriello@xbrl.us (client ID: 69e1257c ...) started the query 
up to 5000 records are found so far ...
 - this set contained fewer than the 5000 possible, only 1 records.
Round 4/291 keyword "http://www.sec.gov/Archives/edgar/data/1551901/00015583

,report.period-end,report.document-type,report.entry-url,report.html-url,report.sec-url,report.base-taxonomy,entity.ticker,report.entity-name,fact
76,2024-06-30,10-Q,http://www.sec.gov/Archives/edgar/data/1000045...,https://www.sec.gov/Archives/edgar/data/100004...,https://www.sec.gov/Archives/edgar/data/100004...,US GAAP 2024,OMCC,"NICHOLAS FINANCIAL, INC.",{'data': [{'dts.target-namespace': 'http://nic...
205,2024-09-30,10-Q/A,http://www.sec.gov/Archives/edgar/data/1000045...,https://www.sec.gov/Archives/edgar/data/100004...,https://www.sec.gov/Archives/edgar/data/100004...,US GAAP 2024,OMCC,OLD MARKET CAPITAL CORPORATION,{'data': [{'dts.target-namespace': 'http://nic...
56,2024-12-31,10-K,http://www.sec.gov/Archives/edgar/data/1000209...,https://www.sec.gov/Archives/edgar/data/100020...,https://www.sec.gov/Archives/edgar/data/100020...,US GAAP 2024,MFIN,MEDALLION FINANCIAL CORP,{'data': [{'dts.target-namespace': 'http://www...
58,2024-12-31,10-K,http://www.sec.gov/Archives/edgar/data/1000209...,https://www.sec.gov/Archives/edgar/data/100020...,https://www.sec.gov/Archives/edgar/data/100020...,US GAAP 2024,MFIN,MEDALLION FINANCIAL CORP,{'data': [{'dts.target-namespace': 'http://www...
96,2024-10-31,10-K,http://www.sec.gov/Archives/edgar/data/1000230...,https://www.sec.gov/Archives/edgar/data/100023...,https://www.sec.gov/Archives/edgar/data/100023...,US GAAP 2024,OCC,OPTICAL CABLE CORPORATION,{'data': [{'dts.target-namespace': 'http://htt...
71,2024-06-30,10-Q,http://www.sec.gov/Archives/edgar/data/1000694...,https://www.sec.gov/Archives/edgar/data/100069...,https://www.sec.gov/Archives/edgar/data/100069...,US GAAP 2024,NVAX,"NOVAVAX, INC.",{'data': [{'dts.target-namespace': 'http://www...
129,2024-06-30,10-Q,http://www.sec.gov/Archives/edgar/data/1001115...,https://www.sec.gov/Archives/edgar/data/100111...,https://www.sec.gov/Archives/edgar/data/100111...,US GAAP 2024,GEOS,GEOSPACE TECHNOLOGIES CORP,{'data': [{'dts.target-namespace': 'http://www...
38,2024-12-31,10-K,http://www.sec.gov/Archives/edgar/data/1001316...,https://www.sec.gov/Archives/edgar/data/100131...,https://www.sec.gov/Archives/edgar/data/100131...,US GAAP 2024,TGTX,"TG THERAPEUTICS, INC.",{'data': [{'dts.target-namespace': 'http://www...
67,2024-12-31,10-K,http://www.sec.gov/Archives/edgar/data/1001385...,https://www.sec.gov/Archives/edgar/data/100138...,https://www.sec.gov/Archives/edgar/data/100138...,US GAAP 2024,NWPX,Northwest Pipe Co.,{'data': [{'dts.target-namespace': 'http://www...
93,2024-12-13,DEF 14A,http://www.sec.gov/Archives/edgar/data/1001907...,https://www.sec.gov/Archives/edgar/data/100190...,https://www.sec.gov/Archives/edgar/data/100190...,US GAAP 2024,ASTC,Astrotech Corporation,{'data': [{'dts.target-namespace': 'http://www...


The next cell can be run to save the dataframe as results.csv and any differences from the `keyword list` as differences.csv

In [4]:
# If you run this program locally, you can save the output to a file
# on your computer (modify D:\results.csv to your system)

df.to_csv(r'D:\DJT\Documents\GitHub\davidtauriello\dqc_us_rules\tests\input\testfiles\testcase-tools\results_2024.csv',sep=',')



### Replace SEC URLs on .travis.yml with roll-forward files


In [ ]:
file_path = "C:/Users/david/Documents/GitHub/DJT/dqc_us_rules/travis.txt"
dict = {
    "https://www.sec.gov/Archives/edgar/data/1001385/000143774923006891/nwpx20221231_10k.htm": "./tests/input/roll-forward/output/srco-20170430_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1004724/000149315223018109/forms-4a.htm": "./tests/input/roll-forward/output/srco-20170731_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1005229/000100522921000155/cmco-20210331.htm": "./tests/input/roll-forward/output/srco-20180731_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1005229/000100522923000021/cmco-20221231.htm": "./tests/input/roll-forward/output/srco-20190131_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1005229/000100522923000169/cmco-20230331.htm": "./tests/input/roll-forward/output/rmhb-20191231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1008586/000149315220011010/strea-20200430.xml": "./tests/input/roll-forward/output/na-20191231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/101382/000156459022006546/umbf-10k_20211231.htm": "./tests/input/roll-forward/output/fulco-20191231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1013880/000155837023002371/ttec-20221231x10k.htm": "./tests/input/roll-forward/output/vcmi-20191231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1015383/000149315223021280/form10-k.htm": "./tests/input/roll-forward/output/fngr-20200229_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1015650/000119312523120018/d408889d20f.htm": "./tests/input/roll-forward/output/bbig-20200331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1017413/000101741323000022/cnq-20221231.htm": "./tests/input/roll-forward/output/gsbc-20200331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1021635/000102163522000037/oge-20211231.htm": "./tests/input/roll-forward/output/fulco-20200331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1021635/000102163522000144/oge-20220930.htm": "./tests/input/roll-forward/output/elst-20190331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1022837/000119312521202577/smfg-20210331.xml": "./tests/input/roll-forward/output/wtnw-20200331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1023459/000168316820002230/simu-20200531.xml": "./tests/input/roll-forward/output/amwd-20210430_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1037038/000103703821000022/rl-20210327.htm": "./tests/input/roll-forward/output/tesi-20200331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1038572/000129281424002600/cbdform20f_2023.htm": "./tests/input/roll-forward/output/snoa-20200331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1039684/000103968423000016/oke-20221231.htm": "./tests/input/roll-forward/output/wint-20200331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1039828/000103982822000022/ael-20211231.htm": "./tests/input/roll-forward/output/lvdw-20200331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1041792/000129281423003055/elpform20fa_2022.htm": "./tests/input/roll-forward/output/frhc-20200331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1041803/000104180320000033/psmt-20200531x10q.htm": "./tests/input/roll-forward/output/rnva-20200331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1043894/000143774923018352/evtn20230331_10k.htm": "./tests/input/roll-forward/output/cbds-20200331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1049659/000129281422002374/sidform20f_2021.htm": "./tests/input/roll-forward/output/ford-20200331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1053352/000155837022002901/htbk-20211231x10k.htm": "./tests/input/roll-forward/output/amtx-20200331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1056358/000105635823000022/mtex-20221231.htm": "./tests/input/roll-forward/output/winmq-20200331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1057706/000105770622000007/fbp0331202210q.htm": "./tests/input/roll-forward/output/vaso-20200331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1066605/000106660520000023/hsii-20200630.htm": "./tests/input/roll-forward/output/invu-20200331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1069308/000156459022029495/acer-10q_20220630.htm": "./tests/input/roll-forward/output/cxdo-20200331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1072379/000141057823000185/nwbo-20221231x10k.htm": "./tests/input/roll-forward/output/mntr-20200331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1075736/000165495420004946/cxdo-20200331.xml": "./tests/input/roll-forward/output/atpc-20200331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1076378/000119312523128248/d401501d20f.htm": "./tests/input/roll-forward/output/na-20200331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/107687/000010768723000015/wgo-20230527.htm": "./tests/input/roll-forward/output/thmg-20200331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1089113/000108911322000002/hsbc-20211231.htm": "./tests/input/roll-forward/output/rmbi-20200331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1090872/000109087222000026/a-20221031.htm": "./tests/input/roll-forward/output/veng-20200331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1092570/000137647420000081/fulo-20191231.htm": "./tests/input/roll-forward/output/muln-20200331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1092570/000137647420000123/fulo-20200331.htm": "./tests/input/roll-forward/output/strm-20200430_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1093672/000165495423003112/pebk_10k.htm": "./tests/input/roll-forward/output/nmex-20200430_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1094324/000157587223001057/sify-20230531.htm": "./tests/input/roll-forward/output/ngio-20200430_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1098009/000118518523001064/amgreat20230630_10k.htm": "./tests/input/roll-forward/output/phmb-20200131_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1101215/000110121523000049/bfh-20221231.htm": "./tests/input/roll-forward/output/abmt-20200430_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1108134/000110813422000004/bhlb-20211231.htm": "./tests/input/roll-forward/output/lake-20200430_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1109357/000110935722000059/exc-20220331.htm": "./tests/input/roll-forward/output/shrg-20200430_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1114446/000161052023000135/ubsag-20230630.htm": "./tests/input/roll-forward/output/apog-20200530_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1118004/000111800422000015/banc-20211231.htm": "./tests/input/roll-forward/output/yubo-20200531_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1119639/000129281422003820/pbrfs2q22usd_6k.htm": "./tests/input/roll-forward/output/simu-20200531_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1122491/000129281423001942/brfform20f_2022.htm": "./tests/input/roll-forward/output/ipii-20200531_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1123596/000143774924022597/babs20240531_10q.htm": "./tests/input/roll-forward/output/psmt-20200531_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1126115/000165495423008433/lzgi_10qa.htm": "./tests/input/roll-forward/output/jctc-20200531_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1130144/000155837022003269/bsrr-20211231x10k.htm": "./tests/input/roll-forward/output/dcac-20200531_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1133421/000113342124000052/noc-20240630.htm": "./tests/input/roll-forward/output/hsii-20200630_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1141688/000149315223016606/form10-q.htm": "./tests/input/roll-forward/output/sui-20200630_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1144546/000168316823003596/hfactor_i10q-033123.htm": "./tests/input/roll-forward/output/both-20200630_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1157557/000129281423002326/cigform20f_2022.htm": "./tests/input/roll-forward/output/andr-20201231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1160330/000129281423001924/bbdform20f_2022.htm": "./tests/input/roll-forward/output/xtgr-20201231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1162896/000149315223016842/form10-q.htm": "./tests/input/roll-forward/output/focs-20201231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1165002/000116500223000033/whg-20221231.htm": "./tests/input/roll-forward/output/rl-20210327_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1170010/000117001023000065/kmx-20230531.htm": "./tests/input/roll-forward/output/thr-20210331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1173643/000117184323004507/exh_991.htm": "./tests/input/roll-forward/output/cmco-20210331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1190370/000173112223001281/e4854_10q.htm": "./tests/input/roll-forward/output/pacx-20210331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1212545/000121254523000093/wal-20221231.htm": "./tests/input/roll-forward/output/smfg-20210331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1230524/000190359623000519/f10q_cyberapps.htm": "./tests/input/roll-forward/output/bpmp-20210930_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1236275/000118518523000161/silversun20221231_10k.htm": "./tests/input/roll-forward/output/cmgo-20200930_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1243429/000124342923000018/mt-20230630.htm": "./tests/input/roll-forward/output/extu-20210331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1274494/000127449423000002/fslr-20221231.htm": "./tests/input/roll-forward/output/cvco-20210403_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1278027/000155837023002392/bgs-20221231x10k.htm": "./tests/input/roll-forward/output/mog-20210403_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1282266/000128226620000031/a2020033110q.htm": "./tests/input/roll-forward/output/poly-20210403_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1286681/000095017023052997/dpz-20230910.htm": "./tests/input/roll-forward/output/dbi-20210501_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1288770/000106299321003175/xtgr-20201231.xml": "./tests/input/roll-forward/output/dlth-20210502_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1299130/000129913023000032/pacb-20221231x10k.htm": "./tests/input/roll-forward/output/trns-20210626_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1300734/000149315223016645/form10-q.htm": "./tests/input/roll-forward/output/wism-20210630_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1300938/000118518522000468/abcoenergy20211231_10k.htm": "./tests/input/roll-forward/output/mplx-20210630_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1304161/000165495420006640/pbsv-20200131.xml": "./tests/input/roll-forward/output/iiin-20210703_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1319947/000131994721000025/dsw-20210501.htm": "./tests/input/roll-forward/output/hrc-20210930_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1321834/000149315223017488/form10-q.htm": "./tests/input/roll-forward/output/cpss-20200331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1326205/000118518523000693/igcpharma20230331_10k.htm": "./tests/input/roll-forward/output/cien-20211030_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1326801/000132680123000013/meta-20221231.htm": "./tests/input/roll-forward/output/dk-20211231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1331520/000133152022000015/homb-20211231.htm": "./tests/input/roll-forward/output/clbk-20211231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1336047/000133604722000030/bwp-20220930.htm": "./tests/input/roll-forward/output/nfsuf-20211231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1338065/000133806523000016/dpm-20230331.htm": "./tests/input/roll-forward/output/up-20211231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1349706/000121465922014094/i111122510q.htm": "./tests/input/roll-forward/output/ubsi-20211231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1350593/000135059323000008/mwa-20221231.htm": "./tests/input/roll-forward/output/tgen-20211231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1358656/000149315222031977/form10-q.htm": "./tests/input/roll-forward/output/ck0001381531-20211231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1360442/000144586620000913/cbds-20200331.xml": "./tests/input/roll-forward/output/sxc-20211231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1364885/000162828023004051/spr-20221231.htm": "./tests/input/roll-forward/output/hsbc-20211231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1367083/000168316820002234/snoa-20200331.xml": "./tests/input/roll-forward/output/amnb-20211231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1368637/000149315223017119/form10-q.htm": "./tests/input/roll-forward/output/sid-20211231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1370755/000095017023004871/tcpc-20221231.htm": "./tests/input/roll-forward/output/oge-20211231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1375793/000149315223018525/form10-q.htm": "./tests/input/roll-forward/output/itri-20211231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1377630/000137763023000061/ncmi-20221229.htm": "./tests/input/roll-forward/output/scpx-20211231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1378789/000137878924000030/aer-20231231.htm": "./tests/input/roll-forward/output/avrw-20211231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1379006/000141057823002146/nnvc-20230630x10k.htm": "./tests/input/roll-forward/output/reno-20211231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1381531/000156459022009469/ufs-10k_20211231.htm": "./tests/input/roll-forward/output/avctq-20211231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1385799/000149315220011580/abmt-20200430.xml": "./tests/input/roll-forward/output/scob-20211231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1391127/000139112723000016/llnw-20221231.htm": "./tests/input/roll-forward/output/abce-20211231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1393781/000166357723000264/qind10q_033123.htm": "./tests/input/roll-forward/output/vuzi-20211231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1415744/000165495420007493/nmex-20200430.xml": "./tests/input/roll-forward/output/adtn-20211231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1425355/000165495423007053/mcvt_10ka.htm": "./tests/input/roll-forward/output/bhlb-20211231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1434601/000171254322000098/tmgi-20220228.xml": "./tests/input/roll-forward/output/angpa-20211231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1434737/000109690623000845/srsg-20221231.htm": "./tests/input/roll-forward/output/homb-20211231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1435812/000151597123000063/arem10q033123.htm": "./tests/input/roll-forward/output/htbk-20211231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1437578/000143757823000005/bfam-20221231.htm": "./tests/input/roll-forward/output/bsrr-20211231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1437925/000147793223004527/gmgi_10q.htm": "./tests/input/roll-forward/output/umbf-20211231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1438461/000164033423001303/elre_10ka.htm": "./tests/input/roll-forward/output/banc-20211231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1440799/000147793223005360/mmex_10k.htm": "./tests/input/roll-forward/output/bpmp-20211231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1463972/000155837022002641/vuzi-20211231x10k.htm": "./tests/input/roll-forward/output/tile-20220102_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1474835/000165495420007737/ipci-20200531.xml": "./tests/input/roll-forward/output/jwn-20220129_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1476034/000155837024010721/mcb-20240630x10q.htm": "./tests/input/roll-forward/output/skil-20220131_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1476963/000155837022003452/htbx-20211231x10k.htm": "./tests/input/roll-forward/output/tmgi-20220228_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1487428/000143774923004919/hrzn20221231_10k.htm": "./tests/input/roll-forward/output/slnh-20220331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1488917/000089710124000422/elmd240883_10k.htm": "./tests/input/roll-forward/output/fbp-20220331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1489096/000148909621000074/thr-20210331.htm": "./tests/input/roll-forward/output/rjac-20220331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1495153/000095017024080082/mmyt-20240331.htm": "./tests/input/roll-forward/output/srax-20220331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1495231/000149523122000140/izea-20220630.htm": "./tests/input/roll-forward/output/exc-20220331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1496383/000166357723000157/ilus10k123122.htm": "./tests/input/roll-forward/output/hmrf-20220430_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1496383/000166357723000293/ilus10qa_033123.htm": "./tests/input/roll-forward/output/flgc-20220630_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1498291/000149315220013326/dcac-20200531.xml": "./tests/input/roll-forward/output/btb-20220630_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1499849/000121390022069303/f20f2022_brasilagro.htm": "./tests/input/roll-forward/output/izea-20220630_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1499961/000143774920011015/nete-20200331.xml": "./tests/input/roll-forward/output/bhp-20220630_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1500305/000149315223018220/form10-q.htm": "./tests/input/roll-forward/output/bpop-20220630_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1503658/000165495420005498/sedh-20200331.xml": "./tests/input/roll-forward/output/grow-20220630_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1514056/000149315220013825/tce-20200331.xml": "./tests/input/roll-forward/output/acer-20220630_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1514705/000151470522000004/sxc-20211231.htm": "./tests/input/roll-forward/output/pbr-20220630_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1517175/000151717523000004/chef-20221230.htm": "./tests/input/roll-forward/output/lnd-20220630_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1520118/000147793223003578/intv_10q.htm": "./tests/input/roll-forward/output/irs-20220630_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1521332/000152133223000013/aptv-20221231.htm": "./tests/input/roll-forward/output/oge-20220930_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1527702/000166357723000243/iqst10q_033123.htm": "./tests/input/roll-forward/output/gpus-20220930_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1537435/000153743522000014/tgen-20211231.htm": "./tests/input/roll-forward/output/hzn-20220930_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1538217/000149315222037214/form10-q.htm": "./tests/input/roll-forward/output/aqua-20220930_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1539680/000106299322014826/form10q.htm": "./tests/input/roll-forward/output/togi-20220930_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1539680/000106299323013910/form10q.htm": "./tests/input/roll-forward/output/bwp-20220930_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1539850/000149315223018647/form10-q.htm": "./tests/input/roll-forward/output/milc-20220930_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1543268/000121390022067720/f20f2022_bitbrother.htm": "./tests/input/roll-forward/output/ctva-20220930_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1546652/000121390023046769/f10q0423_vaneckmerk.htm": "./tests/input/roll-forward/output/a-20221031_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1552000/000155200021000034/mplx-20210630.htm": "./tests/input/roll-forward/output/ncmi-20221229_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1552189/000165495421006233/exdi-20210331.xml": "./tests/input/roll-forward/output/chef-20221230_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1557376/000182912623004245/organicellregen_10q.htm": "./tests/input/roll-forward/output/gsd-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1560452/000118518523000252/trcc20221231_10k.htm": "./tests/input/roll-forward/output/cnq-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1561880/000149315223023943/form10-q.htm": "./tests/input/roll-forward/output/brfs-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1565687/000095017024100587/inta-20240630.htm": "./tests/input/roll-forward/output/cik0001834975-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1568969/000149315223017183/form10-q.htm": "./tests/input/roll-forward/output/srsg-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1576940/000157694023000005/ccs-20221231x10k.htm": "./tests/input/roll-forward/output/trcc-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1590383/000141057822000961/reno-20211231x10k.htm": "./tests/input/roll-forward/output/chnr-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1590715/000147793223003561/arec_10q.htm": "./tests/input/roll-forward/output/mtex-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1593001/000101376223003824/f10k2023_nightfood.htm": "./tests/input/roll-forward/output/aeg-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1599117/000107878220000323/mntr-20200331.xml": "./tests/input/roll-forward/output/trlef-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1602409/000152013820000312/fngr-20200229.xml": "./tests/input/roll-forward/output/mpra-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1603793/000149315224021582/form10-k.htm": "./tests/input/roll-forward/output/whg-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1604643/000160464322000124/aqua-20220930.htm": "./tests/input/roll-forward/output/royl-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1604778/000160477823000055/rfmd-20230401.htm": "./tests/input/roll-forward/output/ypf-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1610520/000161052023000128/ubs-20230630.htm": "./tests/input/roll-forward/output/cig.c-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1624794/000162479421000031/cswi-20210331.htm": "./tests/input/roll-forward/output/aplm-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1627272/000106299323008101/exhibit99-2.htm": "./tests/input/roll-forward/output/nwpx-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1627469/000159991623000158/photozou_10q223o.htm": "./tests/input/roll-forward/output/miti-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1635282/000163528224000030/rmni-20231231.htm": "./tests/input/roll-forward/output/pso-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1637655/000163765522000182/hzn-20220930.htm": "./tests/input/roll-forward/output/rs-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1642365/000182912623004564/alpineautobro_s1.htm": "./tests/input/roll-forward/output/hrzn-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1643301/000147793222001921/curr_10k.htm": "./tests/input/roll-forward/output/ahco-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1644488/000149315220012834/shrv-20200430.xml": "./tests/input/roll-forward/output/cmco-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1648636/000149315223020239/form20-fa.htm": "./tests/input/roll-forward/output/tcpc-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1649744/000164974421000017/dlth-20210502x10q.htm": "./tests/input/roll-forward/output/xpel-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1651052/000155837021001250/focs-20201231x10k.htm": "./tests/input/roll-forward/output/bfh-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1651944/000165194423000010/dmtk-20221231.htm": "./tests/input/roll-forward/output/nwbo-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1653477/000165347723000010/ngvt-20221231.htm": "./tests/input/roll-forward/output/mbi-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1660046/000121390023080084/f20f2023_immuronlimited.htm": "./tests/input/roll-forward/output/qxo-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1661039/000165495423008776/tptw_10q.htm": "./tests/input/roll-forward/output/adt-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1670196/000147793223002932/dswr_10k.htm": "./tests/input/roll-forward/output/nnn-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1670869/000166357720000201/rmhb-20191231.xml": "./tests/input/roll-forward/output/aptv-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1676580/000149315220009097/hccc-20200331.xml": "./tests/input/roll-forward/output/ccs-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1681348/000143774923027411/vpip20230630_20f.htm": "./tests/input/roll-forward/output/ipar-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1690080/000121390023049728/ea179836-s1_180lifesci.htm": "./tests/input/roll-forward/output/edr-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1694426/000169442622000048/dk-20211231.htm": "./tests/input/roll-forward/output/meta-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1694617/000118518523000564/royaleinc20221231_10k.htm": "./tests/input/roll-forward/output/twks-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1703056/000170305623000046/adt-20221231.htm": "./tests/input/roll-forward/output/hhhef-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1704760/000121390022019952/f10k2021_americanvirt.htm": "./tests/input/roll-forward/output/skm-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1706509/000121390023041949/f10q0323_cosmos.htm": "./tests/input/roll-forward/output/elp-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1706509/000121390023044332/f10q0323a1_cosmos.htm": "./tests/input/roll-forward/output/afya-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1708301/000170830121000051/bpmp-20210930.htm": "./tests/input/roll-forward/output/bbdo-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1708301/000170830122000006/bpmp-20211231.htm": "./tests/input/roll-forward/output/pkx-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1711929/000095017024097234/ck0001711929-20240630.htm": "./tests/input/roll-forward/output/cx-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1712178/000095017023050810/naas-20230630.htm": "./tests/input/roll-forward/output/or-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1713210/000149315220013599/aatp-20200331.xml": "./tests/input/roll-forward/output/ttec-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1713909/000101054920000145/wtnw-20200331.xml": "./tests/input/roll-forward/output/wvvi-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1717556/000149315220012166/ednt-20200331.xml": "./tests/input/roll-forward/output/bgs-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1723083/000149315220013056/vcmi-20191231.xml": "./tests/input/roll-forward/output/mwa-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1723596/000172359622000091/clbk-20211231.htm": "./tests/input/roll-forward/output/spr-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1725255/000162828023005590/ahco-20221231.htm": "./tests/input/roll-forward/output/wal-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1729944/000149315223018358/form10-q.htm": "./tests/input/roll-forward/output/fslr-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1739942/000173994223000019/swi-20221231.htm": "./tests/input/roll-forward/output/eml-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1750106/000149315223017508/form10-q.htm": "./tests/input/roll-forward/output/drs-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1755672/000175567222000026/ctva-20220930.htm": "./tests/input/roll-forward/output/ilus-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1755672/000175567223000005/ctva-20221231.htm": "./tests/input/roll-forward/output/swi-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1756640/000149315223016499/form10-q.htm": "./tests/input/roll-forward/output/ppl-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1766363/000095017023005121/edr-20221231.htm": "./tests/input/roll-forward/output/ctva-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1767258/000176725823000012/xpel-20221231.htm": "./tests/input/roll-forward/output/ambc-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1767837/000092708920000320/rmbi-20200331.xml": "./tests/input/roll-forward/output/pebk-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1770450/000177045023000032/xrx-20230331.htm": "./tests/input/roll-forward/output/egio-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1771007/000129281423001938/afyaform20f_2022.htm": "./tests/input/roll-forward/output/gnw-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1774675/000155837022005527/skil-20220131x10k.htm": "./tests/input/roll-forward/output/ugcc-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1777319/000149315224020688/form10-q.htm": "./tests/input/roll-forward/output/oke-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1785493/000149315223021182/form10-q.htm": "./tests/input/roll-forward/output/hayw-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1790169/000147793222007013/flora_6k.htm": "./tests/input/roll-forward/output/pep-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1796160/000109690623001167/qmis-20230331.htm": "./tests/input/roll-forward/output/indb-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1804585/000160706220000180/ngio-20200430.xml": "./tests/input/roll-forward/output/grmc-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1819516/000181951622000009/up-20211231.htm": "./tests/input/roll-forward/output/bfam-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1823575/000095017023047751/zfox-20230731.htm": "./tests/input/roll-forward/output/ngvt-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1829797/000110465921071013/pacx-20210331x10q.htm": "./tests/input/roll-forward/output/alhc-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1831907/000110465923100576/myte-20220630x20f.htm": "./tests/input/roll-forward/output/pacb-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1832161/000149315223018593/form10-q.htm": "./tests/input/roll-forward/output/scl-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1832466/000095017023005096/alhc-20221231.htm": "./tests/input/roll-forward/output/dmtk-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1833756/000162828023009545/drs-20221231.htm": "./tests/input/roll-forward/output/hmnf-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1834622/000183462223000017/hayw-20221231.htm": "./tests/input/roll-forward/output/mcvt-20221231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1834975/000173112223000777/e4627_20-f.htm": "./tests/input/roll-forward/output/apog-20230225_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1838128/000165495423003957/lowlf_10k.htm": "./tests/input/roll-forward/output/lzgi-20230228_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1838431/000121390022020040/f10k2021_sciontechgr2.htm": "./tests/input/roll-forward/output/educ-20230228_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1838876/000168316823003348/geosolar_i10q-033123.htm": "./tests/input/roll-forward/output/dsmd-20230228_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1839341/000162828024035714/core-20240630.htm": "./tests/input/roll-forward/output/mnro-20230325_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1843248/000166357723000279/gsd10k123122.htm": "./tests/input/roll-forward/output/ela-20230331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1846809/000175392623000904/g083559_s4.htm": "./tests/input/roll-forward/output/dcp prc-20230331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1850502/000114036122015609/form10k.htm": "./tests/input/roll-forward/output/kmb-20230331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1852495/000114036122020242/brhc10037514_10q.htm": "./tests/input/roll-forward/output/ambc-20230331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1853047/000101376223003948/f10q0623_hudsonacq1.htm": "./tests/input/roll-forward/output/asb-20230331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1853436/000119312523089818/d456816d10k.htm": "./tests/input/roll-forward/output/apyp-20230331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1862068/000182912623003671/rubicontech_10q.htm": "./tests/input/roll-forward/output/tnfa-20230331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1865127/000101376223003864/f10k2023_lucyscientific.htm": "./tests/input/roll-forward/output/azpn-20230331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1866550/000186655023000008/twks-20221231.htm": "./tests/input/roll-forward/output/aei-20230331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1875547/000149315223036927/form20-f.htm": "./tests/input/roll-forward/output/nvec-20230331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1888846/000159991623000127/ultimate_10q323o.htm": "./tests/input/roll-forward/output/bbls-20230331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1896511/000101376223004045/f20f2022_zkgcnew.htm": "./tests/input/roll-forward/output/otecw-20230331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1897982/000189798223000028/azpn-20230331.htm": "./tests/input/roll-forward/output/spgx-20230331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1901056/000168316823003610/energie_10q-33123.htm": "./tests/input/roll-forward/output/clfd-20230331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1901886/000182912623004746/aurafatprojects_10q.htm": "./tests/input/roll-forward/output/altb-20230331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1930510/000101376223001372/ea185288-6k_vciglobal.htm": "./tests/input/roll-forward/output/mlrt-20230331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1931055/000166357723000285/mdnc10k_022823.htm": "./tests/input/roll-forward/output/iqst-20230331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1931055/000166357723000356/mdnc10q_053123.htm": "./tests/input/roll-forward/output/fcco-20230331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1931055/000166357723000505/mdnc10q_083123.htm": "./tests/input/roll-forward/output/gslr-20230331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1938338/000147793223005302/global_10qa.htm": "./tests/input/roll-forward/output/rhe-20230331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1944885/000119312523183796/d507118df1a.htm": "./tests/input/roll-forward/output/mmmm-20230331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/25743/000119312523145064/d502945d10q.htm": "./tests/input/roll-forward/output/ilus-20230331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/278166/000027816621000029/cvco-20210403.htm": "./tests/input/roll-forward/output/rnva-20230331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/27904/000002790424000009/dal-20240630.htm": "./tests/input/roll-forward/output/qind-20230331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/31107/000165495423002848/eml_10k.htm": "./tests/input/roll-forward/output/edal-20230331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/31667/000118518523000539/edc20230228_10k.htm": "./tests/input/roll-forward/output/tptw-20230331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/318299/000149315220012927/srco-20170430.xml": "./tests/input/roll-forward/output/stcb-20230331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/318299/000149315220012930/srco-20170731.xml": "./tests/input/roll-forward/output/txmd-20230331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/318299/000149315220012938/srco-20180731.xml": "./tests/input/roll-forward/output/arec-20230331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/318299/000149315220012942/srco-20190131.xml": "./tests/input/roll-forward/output/cosg-20230331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/38264/000168316820002062/ford-20200331.xml": "./tests/input/roll-forward/output/cosg-20230331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/38723/000137647420000072/ffc-20191231.xml": "./tests/input/roll-forward/output/global-20230331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/38723/000137647420000115/ffc-20200331.xml": "./tests/input/roll-forward/output/arpc-20230331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/4457/000000445721000040/uhal-20210331.htm": "./tests/input/roll-forward/output/back-20230331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/47518/000004751821000084/hrc-20210930.htm": "./tests/input/roll-forward/output/hwtr-20230331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/55785/000005578523000029/kmb-20230331.htm": "./tests/input/roll-forward/output/sisi-20230331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/59860/000119983523000348/grmc-10k.htm": "./tests/input/roll-forward/output/intv-20230331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/64463/000175392622000723/g083007_10q.htm": "./tests/input/roll-forward/output/vipz-20230331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/6494/000120677421000940/andr-20201231.xml": "./tests/input/roll-forward/output/sify-20230331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/66600/000149315223018572/form10-q.htm": "./tests/input/roll-forward/output/elre-20230331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/67887/000006788721000029/mog-20210403.htm": "./tests/input/roll-forward/output/evtn-20230331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/6845/000000684520000019/apog-202053010xq.htm": "./tests/input/roll-forward/output/lcut-20230331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/6845/000000684523000006/apog-20230225.htm": "./tests/input/roll-forward/output/lark-20230331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/701719/000165495423005744/ela_10q.htm": "./tests/input/roll-forward/output/cmco-20230331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/704562/000168316823004329/avid_i10k-043023.htm": "./tests/input/roll-forward/output/igc-20230331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/711034/000105291820000126/thmg-20200331.xml": "./tests/input/roll-forward/output/atnf-20230331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/715787/000071578722000005/tile-20220102.htm": "./tests/input/roll-forward/output/none-20230331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/72333/000007233322000060/jwn-20220129.htm": "./tests/input/roll-forward/output/xrx-20230331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/723603/000095017024083115/culp-20240428.htm": "./tests/input/roll-forward/output/prop-20230331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/724910/000137647423000283/nvec-20230331.htm": "./tests/input/roll-forward/output/poww-20230331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/729986/000119312522061038/d233531d10k.htm": "./tests/input/roll-forward/output/runicorncom-20230331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/738214/000165495420005416/amtx-20200331.xml": "./tests/input/roll-forward/output/qmis-20230331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/741516/000143774922006107/americannb20211231_10k.htm": "./tests/input/roll-forward/output/qrvo-20230401_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/751364/000095017023002248/nnn-20221231.htm": "./tests/input/roll-forward/output/gmgi-20230430_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/752294/000105291820000090/elst-20190331.xml": "./tests/input/roll-forward/output/mmex-20230430_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/754811/000143774923015310/usglobal20230307_10ka.htm": "./tests/input/roll-forward/output/hmrf-20230430_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/763901/000119312522216145/d278557d10q.htm": "./tests/input/roll-forward/output/cyap-20230430_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/764401/000143774921017350/iiin20210703_10q.htm": "./tests/input/roll-forward/output/pxpc-20230430_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/769218/000119312523076544/d408064d20f.htm": "./tests/input/roll-forward/output/ounz-20230430_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/77476/000007747623000007/pep-20221231.htm": "./tests/input/roll-forward/output/cdmo-20230430_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/776901/000077690123000064/indb-20221231.htm": "./tests/input/roll-forward/output/ocel-20230430_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/7789/000000778923000035/asb-20230331.htm": "./tests/input/roll-forward/output/pvh-20230430_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/780571/000078057122000004/itri-20211231.htm": "./tests/input/roll-forward/output/uhgi-20230430_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/78239/000007823923000091/pvh-20230430.htm": "./tests/input/roll-forward/output/ivdn-20230430_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/788920/000107997323001429/pdex_10k-063023.htm": "./tests/input/roll-forward/output/wgo-20230527_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/793628/000107997323000705/chnr_20f-123122.htm": "./tests/input/roll-forward/output/zkgc-20230531_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/796505/000117184323003018/clfd20230331_10q.htm": "./tests/input/roll-forward/output/trx-20230531_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/798081/000165495420006426/lake-20200430.xml": "./tests/input/roll-forward/output/kmx-20230531_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/802257/000118518523000727/mitesco20221231_10k.htm": "./tests/input/roll-forward/output/ptzh-20230531_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/808326/000080832624000030/emkr-20240630.htm": "./tests/input/roll-forward/output/afar-20230531_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/811809/000119312522238375/d324080d20f.htm": "./tests/input/roll-forward/output/dsmd-20230531_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/814585/000119312523054064/d387374d10k.htm": "./tests/input/roll-forward/output/deo-20230630_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/822663/000175392623000213/g083417_10k.htm": "./tests/input/roll-forward/output/frhc-20230630_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/825171/000160706223000281/hhhef123122form20f.htm": "./tests/input/roll-forward/output/huda r-20230630_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/835403/000083540323000016/deo-20230630.htm": "./tests/input/roll-forward/output/aagh-20230630_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/838875/000119983523000177/wvvi-10k.htm": "./tests/input/roll-forward/output/vcig-20230630_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/839087/000165495420006323/vaso-20200331.xml": "./tests/input/roll-forward/output/mt-20230630_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/854560/000092708920000286/gsbc-20200331.htm": "./tests/input/roll-forward/output/vvpr-20230630_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/861884/000155837023002368/rs-20221231x10k.htm": "./tests/input/roll-forward/output/naas-20230630_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/862651/000149315220013684/invu-20200331.xml": "./tests/input/roll-forward/output/ubsag-20230630_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/874396/000162828023017221/lcut-20230331.htm": "./tests/input/roll-forward/output/ubs-20230630_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/874501/000087450123000040/ambc-20221231.htm": "./tests/input/roll-forward/output/pdex-20230630_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/874501/000087450123000092/ambc-20230331.htm": "./tests/input/roll-forward/output/mvco-20230630_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/876427/000156276223000263/mnro-20230325x10k.htm": "./tests/input/roll-forward/output/lsdi-20230630_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/885307/000121716020000041/jctcf-20200531.xml": "./tests/input/roll-forward/output/nnvc-20230630_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/889132/000119312523123987/d399110d20f.htm": "./tests/input/roll-forward/output/ngtf-20230630_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/889609/000168316820001394/cpss-20200331.xml": "./tests/input/roll-forward/output/lcfyw-20230630_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/894560/000155116320000061/both-20200630.xml": "./tests/input/roll-forward/output/myte-20220630_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/895464/000119380520000864/magaa-20200531.xml": "./tests/input/roll-forward/output/imrn-20230630_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/896493/000121465923005425/r4523110qa1sept2022.htm": "./tests/input/roll-forward/output/zfox-20230731_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/904851/000119312523097822/d357382d20f.htm": "./tests/input/roll-forward/output/dsmd-20230831_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/912593/000091259320000137/sui-20200630.htm": "./tests/input/roll-forward/output/dpz-20230910_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/914025/000091402521000019/plt-20210403.htm": "./tests/input/roll-forward/output/rmni-20231231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/921183/000143774923005328/hmnf20221231_10k.htm": "./tests/input/roll-forward/output/aer-20231231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/922224/000092222423000010/ppl-20221231.htm": "./tests/input/roll-forward/output/cbd-20231231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/924095/000149315223037175/form10-k.htm": "./tests/input/roll-forward/output/iwpt-20240229_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/924805/000092480523000104/frhc-20230630.htm": "./tests/input/roll-forward/output/ciso-20240331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/924805/000165495420007646/frhc-20200331.xml": "./tests/input/roll-forward/output/mmyt-20240331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/926282/000095017022002054/adtn-20211231.htm": "./tests/input/roll-forward/output/culp-20240428_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/931059/000149315220013561/rnva-20200331.xml": "./tests/input/roll-forward/output/babb-20240531_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/931059/000149315223017256/form10-q.htm": "./tests/input/roll-forward/output/dal-20240630_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/932781/000155278123000271/e23251_fcco-10q.htm": "./tests/input/roll-forward/output/corz-20240630_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/933267/000165495422014187/irsa_20f.htm": "./tests/input/roll-forward/output/emkr-20240630_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/936395/000093639521000054/cien-20211030.htm": "./tests/input/roll-forward/output/noc-20240630_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/938323/000119312523086885/d429723d20f.htm": "./tests/input/roll-forward/output/elmd-20240630_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/94049/000095017023004933/scl-20221231.htm": "./tests/input/roll-forward/output/ck0001711929-20240630_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/946486/000143774920011216/wint-20200331.xml": "./tests/input/roll-forward/output/inta-20240630_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/99302/000120677421001956/transcat3937691-10q.htm": "./tests/input/roll-forward/output/mcb-20240630_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1029800/000102980021000016/form10k2020.htm": "./tests/input/roll-forward/output/bbls-20190930_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1037540/000165642322000013/bxp-20211231.htm": "./tests/input/roll-forward/output/cwk-20200630_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1045810/000104581020000189/nvda-20201025.htm": "./tests/input/roll-forward/output/trnf-20200630_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1135951/000157587221000138/rdy-20210331.xml": "./tests/input/roll-forward/output/eaco-20200831_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1207179/000120717922000020/glng-20220630_htm.xml": "./tests/input/roll-forward/output/esmc-20200930_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1316644/000118518520001620/aero-20200930.xml": "./tests/input/roll-forward/output/amct-20200930_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1346655/000101738620000468/cmgo-20200930.xml": "./tests/input/roll-forward/output/gure-20200930_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1368458/000156459020054910/sbh-10k_20200930.htm": "./tests/input/roll-forward/output/nutx-20200930_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1368637/000149315221013015/bbls-20190930.xml": "./tests/input/roll-forward/output/aero-20200930_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1401395/000156459021036805/nept-20210331.xml": "./tests/input/roll-forward/output/cswi-20210331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1404281/000156459020054210/nvus-20200930.xml": "./tests/input/roll-forward/output/eldn-20200930_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1437491/000121390021034916/zest-20210331.xml": "./tests/input/roll-forward/output/pyx-20210331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1479681/000160706220000329/clnh-20200930.xml": "./tests/input/roll-forward/output/issc-20200930_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1552033/000155203321000008/ck0001552033-20201231.htm": "./tests/input/roll-forward/output/fami-20200930_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1603145/000160314520000046/nep-20200630_htm.xml": "./tests/input/roll-forward/output/hwm-20200930_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1615346/000156459021040509/lmrk-10q_20210630_htm.xml": "./tests/input/roll-forward/output/iec-20200930_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1626556/000110465920125690/amct-20200930.xml": "./tests/input/roll-forward/output/j-20201002_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1628369/000162836920000035/cwk-20200630.htm": "./tests/input/roll-forward/output/nvda-20201025_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1631569/000163156921000041/chct-20210630.htm": "./tests/input/roll-forward/output/amat-20201025_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1701261/000110465921009423/fami-20200930.xml": "./tests/input/roll-forward/output/sbh-20200930_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1721741/000149315221015375/lazy-20210331.xml": "./tests/input/roll-forward/output/smtc-20201025_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1774155/000114036121022585/btrs-20210628.xml": "./tests/input/roll-forward/output/apog-20201128_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/1778805/000149315220016224/trnf-20200630.xml": "./tests/input/roll-forward/output/nruc-20201130_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/3545/000156459021059499/alco-10k_20210930.htm": "./tests/input/roll-forward/output/tru-20201231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/4281/000000428120000185/arnc-20200930.htm": "./tests/input/roll-forward/output/rdy-20210331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/49728/000004972820000089/iec-20200930.htm": "./tests/input/roll-forward/output/nept-20210331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/52988/000005298820000070/jec-20201002.htm": "./tests/input/roll-forward/output/gorv-20210331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/6845/000000684521000003/apog-20201128.htm": "./tests/input/roll-forward/output/cik0000926042-20210331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/6951/000000695120000048/amat-20201025.htm": "./tests/input/roll-forward/output/uba-20201031_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/70502/000007050221000009/nru-20201130.htm": "./tests/input/roll-forward/output/btrs-20210628_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/784539/000110465920130448/eaco-20200831.xml": "./tests/input/roll-forward/output/ecar-20210331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/794619/000079461921000046/amwd-20210430.htm": "./tests/input/roll-forward/output/uhal-20210331_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/836690/000110465920138570/issc-20200930.xml": "./tests/input/roll-forward/output/bbva-20210630_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/842180/000084218021000019/bbva-20210630.htm": "./tests/input/roll-forward/output/chct-20210630_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/862668/000086266820000013/esmc-20200930.xml": "./tests/input/roll-forward/output/ck0000932782-20211031_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/885462/000119380520001433/gure-20200930.xml": "./tests/input/roll-forward/output/alco-20210930_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/88941/000008894120000013/smtc-20201025.htm": "./tests/input/roll-forward/output/bxp-20211231_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/926042/000119312521202050/ttm-20210331.xml": "./tests/input/roll-forward/output/nep-20200630_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/932782/000119312521348640/d231607d6k.htm": "./tests/input/roll-forward/output/lmrk-20210630_htm.xml",
    "https://www.sec.gov/Archives/edgar/data/939930/000093993021000034/pyx-20210331.htm": "./tests/input/roll-forward/output/glng-20220630_htm.xml"
}

with open(file_path, 'r+') as file:
    content = file.read()
    for key, value in dict.items():
        content = content.replace(key, value)
    file.seek(0)
    file.truncate()
    file.write(content)